In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("silver_schema", "silver", "2. Silver Schema")
CATALOG = dbutils.widgets.get("catalog_name")
SILVER = dbutils.widgets.get("silver_schema")

TABLE = f"{CATALOG}.{SILVER}.streets_business"
DEMO_STREET_ID = 1  # arbitrary — any valid street_id works for this demo

print(f"Table: {TABLE}")

## Step 1 — Baseline Audit (state before any change)

In [0]:
baseline = spark.sql(f"""
    SELECT COUNT(*) AS row_count, ROUND(AVG(pollution), 4) AS avg_pollution
    FROM {TABLE}
    WHERE street_id = {DEMO_STREET_ID}
""").collect()[0]
print(f"\n[Baseline] street_id={DEMO_STREET_ID} -> rows={baseline['row_count']:,}, "
      f"avg_pollution={baseline['avg_pollution']}")

delta_table = DeltaTable.forName(spark, TABLE)
version_before = delta_table.history(1).collect()[0]["version"]
print(f"[Baseline] current table version: {version_before}")

## Step 2 — Idempotency check: has this demo already been applied?

In [0]:
already_applied = (delta_table.history()
    .filter(F.col("operation") == "UPDATE")
    .count() > 0)

if already_applied:
    prior_update_version = (delta_table.history()
        .filter(F.col("operation") == "UPDATE")
        .orderBy("version")
        .first()["version"])
    print(f"\nAn UPDATE already exists in history (version {prior_update_version}) — "
          f"skipping to avoid compounding the correction. Delete streets_business and "
          f"rerun 11_silver_business_rules.py first if you want a clean demo run.")
else:
    # ── Step 3 — Transactional Update (Atomicity) ──────────────────────────
    # Demo scenario (illustrative, not a real finding from the data): the
    # sensor on street_id=1 is treated as miscalibrated, reporting pollution
    # 5% high — apply a corrective adjustment. Delta guarantees this UPDATE
    # is all-or-nothing: if it fails partway through, no row is left changed.
    print(f"\n[Update] Applying demo correction to street_id={DEMO_STREET_ID}...")
    spark.sql(f"""
        UPDATE {TABLE}
        SET pollution = ROUND(pollution * 0.95, 6)
        WHERE street_id = {DEMO_STREET_ID}
    """)
    print("[Update] Complete.")

## Step 4 — Lineage & History (Audit Trail / Durability)

In [0]:
version_after = delta_table.history(1).collect()[0]["version"]
print(f"\n[History] version before={version_before}, version after={version_after}")

history_df = (delta_table.history()
    .select("version", "timestamp", "operation", "operationParameters")
    .orderBy(F.col("version").desc())
    .limit(5))
display(history_df)

## Step 5 — Time Travel: compare state at version_before vs current

In [0]:
after_state = spark.sql(f"""
    SELECT COUNT(*) AS row_count, ROUND(AVG(pollution), 4) AS avg_pollution
    FROM {TABLE}
    WHERE street_id = {DEMO_STREET_ID}
""").collect()[0]

comparison = spark.sql(f"""
    SELECT 'Before (VERSION AS OF {version_before})' AS state,
           ROUND(AVG(pollution), 4) AS avg_pollution
    FROM {TABLE} VERSION AS OF {version_before}
    WHERE street_id = {DEMO_STREET_ID}

    UNION ALL

    SELECT 'Current (version {version_after})' AS state,
           ROUND(AVG(pollution), 4) AS avg_pollution
    FROM {TABLE}
    WHERE street_id = {DEMO_STREET_ID}
""")

print(f"\n{'='*60}\n  TIME TRAVEL EVIDENCE — street_id={DEMO_STREET_ID}\n{'='*60}")
display(comparison)
print(f"{'='*60}")
print(f"  Old avg_pollution (version {version_before}): captured above")
print(f"  New avg_pollution (version {version_after}) : captured above")
print(f"  This comparison is reproducible on ANY run — version numbers were")
print(f"  read from DESCRIBE HISTORY at runtime, not hardcoded.")
print(f"{'='*60}")